In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
from scipy import stats

def coverage_and_width(low, up, y_test):
    width = up - low
    coverage = np.mean((low <= y_test) & (y_test <= up))
    return width.mean(), coverage

def calculate_performance(y_pred, y_test):
    if np.isnan(y_pred).any() or np.isnan(y_test).any():
        mask = ~np.isnan(y_pred) & ~np.isnan(y_test)
        y_pred = y_pred[mask]
        y_test = y_test[mask]
    mse = np.mean((y_pred - y_test) ** 2)
    mae = np.mean(np.abs(y_pred - y_test))
    rho = stats.spearmanr(y_pred, y_test)[0]
    tau = stats.kendalltau(y_pred, y_test)[0]
    return mse, mae, rho, tau


Mounted at /content/drive


In [ ]:
print(df.head())

   index  low  up  init_score_raw  init_score_weight  re_score_raw  \
0      0    1   3               2           1.655164           2.0   
1      1    1   4               3           3.047397           3.0   
2      2    2   5               4           3.994307           4.0   
3      3    2   5               4           3.995161           4.0   
4      4    1   3               2           1.828459           2.0   

  re_score_weight  ground_truth  
0    [2.02178899]           1.0  
1    [3.01226609]           2.0  
2    [3.92575974]           4.0  
3    [3.95181344]           5.0  
4    [2.03532003]           1.0  


In [ ]:
dimensions = ['cosmos', 'drop', 'esnli', 'gsm8k']

all_results = []

for dimension in dimensions:
    file_path = f'bestseed_reprompt_socreval_{dimension}.csv'
    df = pd.read_csv(file_path)
    df = df.iloc[:, :-1]

    low = df.iloc[:, 1].to_numpy().astype(np.float32)
    up = df.iloc[:, 2].to_numpy().astype(np.float32)
    y_test = df.iloc[:, -1].to_numpy().astype(np.float32)
    init_score_raw  = df.iloc[:, 3].to_numpy().astype(np.float32)
    init_score_weight = df.iloc[:, 4].to_numpy().astype(np.float32)
    re_score_raw = df.iloc[:, 5].to_numpy().astype(np.float32)
    re_score_weight = [float(item.strip('[]')) for item in df.iloc[:, 6]]

    width_init, coverage_init = coverage_and_width(low, up, y_test)

    results = {
        "Dataset": dimension,
        "Method": [],
        "MSE": [],
        "MAE": [],
        "Spearman ρ": [],
        "Kendall τ": [],
        "Width": [],
        "Coverage": []
    }

    mse, mae, rho, tau = calculate_performance(init_score_raw, y_test)
    results["Method"].append("Initial Raw")
    results["MSE"].append(mse)
    results["MAE"].append(mae)
    results["Spearman ρ"].append(rho)
    results["Kendall τ"].append(tau)
    results["Width"].append(width_init)
    results["Coverage"].append(coverage_init)

    mse, mae, rho, tau = calculate_performance(re_score_raw, y_test)
    results["Method"].append("Reprompt Raw")
    results["MSE"].append(mse)
    results["MAE"].append(mae)
    results["Spearman ρ"].append(rho)
    results["Kendall τ"].append(tau)
    results["Width"].append(width_init)
    results["Coverage"].append(coverage_init)

    mse, mae, rho, tau = calculate_performance(init_score_weight, y_test)
    results["Method"].append("Initial Weighted")
    results["MSE"].append(mse)
    results["MAE"].append(mae)
    results["Spearman ρ"].append(rho)
    results["Kendall τ"].append(tau)
    results["Width"].append(width_init)
    results["Coverage"].append(coverage_init)

    mse, mae, rho, tau = calculate_performance(re_score_weight, y_test)
    results["Method"].append("Reprompt Weighted")
    results["MSE"].append(mse)
    results["MAE"].append(mae)
    results["Spearman ρ"].append(rho)
    results["Kendall τ"].append(tau)
    results["Width"].append(width_init)
    results["Coverage"].append(coverage_init)

    for i in range(len(results["Method"])):
        all_results.append({
            "Dataset": results["Dataset"],
            "Method": results["Method"][i],
            "MSE": results["MSE"][i],
            "MAE": results["MAE"][i],
            "Spearman ρ": results["Spearman ρ"][i],
            "Kendall τ": results["Kendall τ"][i],
            "Width": results["Width"][i],
            "Coverage": results["Coverage"][i]
        })

all_results_df = pd.DataFrame(all_results)
print(all_results_df)

   Dataset             Method       MSE       MAE  Spearman ρ  Kendall τ  \
0   cosmos        Initial Raw  2.204082  1.163265    0.480293   0.419364   
1   cosmos       Reprompt Raw  2.193877  1.153061    0.476310   0.417798   
2   cosmos   Initial Weighted  2.052884  1.133203    0.508314   0.390947   
3   cosmos  Reprompt Weighted  2.111918  1.167847    0.499106   0.377264   
4     drop        Initial Raw  1.371429  0.800000    0.603949   0.551028   
5     drop       Reprompt Raw  1.380952  0.809524    0.603821   0.550921   
6     drop   Initial Weighted  1.333399  0.800079    0.612075   0.485937   
7     drop  Reprompt Weighted  1.345206  0.814889    0.634605   0.503989   
8    esnli        Initial Raw  0.684211  0.631579    0.561363   0.517585   
9    esnli       Reprompt Raw  0.657895  0.631579    0.595460   0.548320   
10   esnli   Initial Weighted  0.610842  0.623154    0.639802   0.512257   
11   esnli  Reprompt Weighted  0.605095  0.638822    0.646223   0.517462   
12   gsm8k  